In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = 224
BATCH_SIZE = 32

# กำหนดคลาสที่ต้องการใช้งาน (ข้ามสารกาแฟดิบ หรือ Green)
TARGET_CLASSES = ['Light', 'Medium', 'Dark']
NUM_CLASSES = len(TARGET_CLASSES)

# โหลดชุดข้อมูลจากโฟลเดอร์ train โดยตรง
train_dataset = tf.keras.utils.image_dataset_from_directory(
    '../datasets/train', # เปลี่ยน Path ให้ตรงกับโฟลเดอร์จริง
    labels='inferred',
    class_names=TARGET_CLASSES,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

# โหลดชุดข้อมูลจากโฟลเดอร์ test เพื่อใช้เป็น Validation
val_dataset = tf.keras.utils.image_dataset_from_directory(
    '../datasets/test', # เปลี่ยน Path ให้ตรงกับโฟลเดอร์จริง
    labels='inferred',
    class_names=TARGET_CLASSES,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

print(f"คลาสที่โหลดมาใช้งาน: {train_dataset.class_names}")

# Data Augmentation: สุ่มหมุน พลิก และปรับแสง
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.15), # สุ่มซูมเข้า/ออก 15%
])

Found 900 files belonging to 3 classes.
Found 300 files belonging to 3 classes.
คลาสที่โหลดมาใช้งาน: ['Light', 'Medium', 'Dark']


In [5]:
# 1. โหลด MobileNetV2 แบบดึงค่าน้ำหนักจาก ImageNet (ฉลาดที่สุด)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # ล็อกค่าน้ำหนักไว้ก่อน

# 2. ประกอบโครงสร้าง
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)

# แปลงค่าพิกเซลจาก 0-255 ให้อยู่ในช่วง -1 ถึง 1 (ตามที่ MobileNetV2 ต้องการเป๊ะๆ)
x = layers.Rescaling(1./127.5, offset=-1)(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

# คอมไพล์ด้วย Learning rate มาตรฐาน
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,261,827 (8.63 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
# หยุดเทรนอัตโนมัติถ้าค่า val_loss ไม่ลดลง 3 รอบติดกัน และดึงค่าน้ำหนักรอบที่ดีที่สุดกลับมา
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)

print("เริ่มการฝึกแบบจำลอง...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50, # เพิ่มรอบสูงสุดเป็น 50
    callbacks=[early_stop]
)

model.save('../src/coffee_roast_model.keras')
print("เทรนเสร็จสิ้น และบันทึกโมเดล .keras สำเร็จ")

เริ่มการฝึกแบบจำลอง...
Epoch 1/50


d:\Projects\Roasted-Coffee-Classifier\env\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


29/29 ━━━━━━━━━━━━━━━━━━━━ 20s 589ms/step - accuracy: 0.6444 - loss: 0.7914 - val_accuracy: 0.8567 - val_loss: 0.4240
Epoch 2/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 15s 525ms/step - accuracy: 0.8722 - loss: 0.3559 - val_accuracy: 0.8867 - val_loss: 0.2942
Epoch 3/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 16s 549ms/step - accuracy: 0.9056 - loss: 0.2613 - val_accuracy: 0.9100 - val_loss: 0.2393
Epoch 4/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 16s 540ms/step - accuracy: 0.9278 - loss: 0.2266 - val_accuracy: 0.9367 - val_loss: 0.1898
Epoch 5/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 16s 560ms/step - accuracy: 0.9333 - loss: 0.1758 - val_accuracy: 0.9433 - val_loss: 0.1716
Epoch 6/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 16s 549ms/step - accuracy: 0.9256 - loss: 0.1730 - val_accuracy: 0.9567 - val_loss: 0.1506
Epoch 7/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 15s 531ms/step - accuracy: 0.9511 - loss: 0.1473 - val_accuracy: 0.9633 - val_loss: 0.1383
Epoch 8/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 15s 528ms/step - accuracy: 0.9644 - loss: 0.1329 - val_accuracy: 0.963

In [7]:
print("กำลังแปลงไฟล์เป็น TensorFlow Lite สำหรับ Raspberry Pi...")

# ดึงโมเดลที่ฝึกเสร็จแล้วมาทำการแปลง
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# สั่งบีบอัดโมเดล (Post-training Quantization) 
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

# บันทึกไฟล์ที่ถูกบีบอัดแล้ว
with open('../src/coffee_roast_model_quantized.tflite', 'wb') as f:
    f.write(tflite_model)
    
print("แปลงไฟล์ .tflite สำเร็จ! นำไฟล์นี้ไปรันบนบอร์ด Pi ได้เลย")

กำลังแปลงไฟล์เป็น TensorFlow Lite สำหรับ Raspberry Pi...
INFO:tensorflow:Assets written to: C:\Users\ASUS\AppData\Local\Temp\tmphcpt1gr8\assets


INFO:tensorflow:Assets written to: C:\Users\ASUS\AppData\Local\Temp\tmphcpt1gr8\assets


Saved artifact at 'C:\Users\ASUS\AppData\Local\Temp\tmphcpt1gr8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_171')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  1939158465040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1937196638544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1939158463312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1937196638352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1937196638736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1937196633168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1937196637776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1937196637968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1937196637584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1937196639120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  19